# NB8 — Mitochondrial QC Threshold Sensitivity

Re-filters each microglia cohort at four mitochondrial read fraction thresholds
(MT% <= 20%, 25%, 30%, 35%) and recomputes MES-tolerance-positioning Spearman correlations.
This confirms that the 20% threshold applied in the main quality-control pipeline is not
a critical parameter.

**Paper:** Zafar SA, Qin W, Liu C, Khan AA, Faisal MS.
*Thymus-derived myeloid programs track microglial tolerance states across human cohorts.* (2026).

**Runs after:** NB4/NB5 (for scored microglia .h5ad files with MES0X_score columns).

**Outputs:**
- `Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx` — long-form correlations and pivot table
- `Supplementary_FigS24_Mito_Threshold_Sensitivity.png` — heatmap per cohort

> Set `MES_BASE_DIR` to your project root before running.

In [ ]:
from __future__ import annotations

import os, time, gc, warnings, traceback
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.stats as scipy_stats
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.rcParams.update({
    "font.family":        "Arial",
    "font.size":          8,
    "axes.titlesize":     9,
    "axes.labelsize":     8,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "legend.fontsize":    7,
    "figure.dpi":         150,
    "savefig.dpi":        1200,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "axes.linewidth":     0.8,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_style('ticks')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

import anndata as ad
import scanpy as sc

try:
    from statsmodels.stats.multitest import multipletests
    HAS_SM = True
except ImportError:
    HAS_SM = False


# 0) Paths

np.random.seed(42)

BASE_DIR  = Path(os.environ.get("MES_BASE_DIR", "."))  # <-- SET PATH
PROC_DIR  = BASE_DIR / "Process Data" / "aim2_microglia"
MANUS_DIR_CANDIDATES = [BASE_DIR / "Manuscript data", BASE_DIR / "Manuscript Data"]
MANUS_DIR = next((p for p in MANUS_DIR_CANDIDATES if p.exists()), MANUS_DIR_CANDIDATES[0])
TAB_DIR   = MANUS_DIR / "Tables"
FIG_DIR   = MANUS_DIR / "Figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SCORED_GLOB     = str(PROC_DIR / "*__microglia_scored.h5ad")
MITO_THRESHOLDS = [20, 25, 30, 35]
MES_COLS        = [f"MES0{i}" for i in range(1, 9)]
DPI             = 1200
N_BOOT          = 300
MIN_CELLS       = 100


# 1) Utilities

def _now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str) -> None:
    print(f"[{_now()}] {msg}", flush=True)

def save_fig(path: Path, fig=None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    (fig or plt).savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig) if fig is not None else plt.close()
    log(f"[SAVED FIG] {path}")

def save_xlsx(dfs: Dict[str, pd.DataFrame], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in dfs.items():
            if df is not None and not df.empty:
                df.to_excel(w, index=False, sheet_name=name[:31])
    log(f"[SAVED TABLE] {path}")

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    if HAS_SM:
        _, q, _, _ = multipletests(pvals, method="fdr_bh")
        return q
    n = len(pvals)
    order = np.argsort(pvals)
    q = np.empty(n)
    q[order] = pvals[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    return np.minimum(q, 1.0)

def spearman_ci(x, y, n_boot=N_BOOT, seed=42) -> Dict:
    x = pd.to_numeric(x, errors="coerce").to_numpy()
    y = pd.to_numeric(y, errors="coerce").to_numpy()
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)
    if n < 5:
        return {"r": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "p": np.nan, "n": n}
    r_obs, p_obs = scipy_stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boots = np.array([
        scipy_stats.spearmanr(x[idx := rng.integers(0, n, n)], y[idx])[0]
        for _ in range(n_boot)
    ])
    boots = boots[np.isfinite(boots)]
    ci_lo = float(np.percentile(boots, 2.5))  if len(boots) else np.nan
    ci_hi = float(np.percentile(boots, 97.5)) if len(boots) else np.nan
    return {"r": float(r_obs), "ci_lo": ci_lo, "ci_hi": ci_hi, "p": float(p_obs), "n": n}

def detect_donor_col(obs: pd.DataFrame) -> Optional[str]:
    for c in ['donor_id','donor','subject','SubjectID','subject_id',
              'individualID','individual_id','case_id','participant_id']:
        if c in obs.columns and 2 <= obs[c].nunique() <= 500:
            return c
    return None

def donor_mean(obs: pd.DataFrame, donor_col: str, cols: List[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in obs.columns]
    return obs.groupby(donor_col)[cols].mean().reset_index()

def ensure_pct_mt(adata: ad.AnnData) -> bool:
    if 'pct_counts_mt' in adata.obs.columns:
        return True
    try:
        adata.var['mt'] = adata.var_names.astype(str).str.upper().str.startswith('MT-')
        sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
        return 'pct_counts_mt' in adata.obs.columns
    except Exception as e:
        log(f"  Could not compute pct_counts_mt: {e}")
        return False

TOL_H = ["P2RY12","CX3CR1","TMEM119","GPR34","SALL1","CSF1R","OLFML3"]
TOL_A = ["APOE","SPP1","LPL","TREM2","CST7","CTSD","TYROBP","FCER1G","LGALS3","CD68"]

def ensure_tolerance(adata: ad.AnnData) -> None:
    if 'tolerance_positioning' not in adata.obs.columns:
        vu = adata.var_names.astype(str).str.upper()
        u2o = dict(zip(vu, adata.var_names))
        def score(genes, key):
            present = [u2o[g] for g in genes if g in u2o]
            if len(present) >= 3:
                sc.tl.score_genes(adata, present, score_name=key, use_raw=False)
        score(TOL_H, '_th8')
        score(TOL_A, '_ta8')
        if '_th8' in adata.obs and '_ta8' in adata.obs:
            adata.obs['tolerance_positioning'] = adata.obs['_th8'] - adata.obs['_ta8']


# 2) Load scored microglia cohorts

log("NB8: Mitochondrial QC threshold sensitivity")

import glob as _glob
h5ads = sorted(_glob.glob(SCORED_GLOB))
assert len(h5ads) >= 4, f'Expected >=4 scored .h5ad files, found {len(h5ads)}. Re-run NB4/NB5 first.'

cohorts: Dict[str, ad.AnnData] = {}
for fp in h5ads:
    name = Path(fp).stem.replace('__microglia_scored', '')
    log(f"  {name}")
    cohorts[name] = sc.read_h5ad(fp)
    gc.collect()
log(f"Loaded {len(cohorts)} cohorts")


# 3) Re-filter at each MT% threshold and recompute correlations

log(f"Testing thresholds: {MITO_THRESHOLDS}%")
rows = []

for ds, adata in cohorts.items():
    log(f"  {ds}")
    if not ensure_pct_mt(adata):
        continue
    ensure_tolerance(adata)
    if 'tolerance_positioning' not in adata.obs.columns:
        continue

    mes_score_cols = [f"{m}_score" for m in MES_COLS]
    mes_score_cols = [c for c in mes_score_cols if c in adata.obs.columns]
    if not mes_score_cols:
        continue

    donor_col = detect_donor_col(adata.obs)
    mt_pct    = pd.to_numeric(adata.obs['pct_counts_mt'], errors='coerce').to_numpy()

    for thr in MITO_THRESHOLDS:
        keep   = (mt_pct <= thr) & np.isfinite(mt_pct)
        n_kept = int(keep.sum())
        if n_kept < MIN_CELLS:
            continue

        sub = adata.obs.loc[keep].copy()
        score_cols = ['tolerance_positioning'] + mes_score_cols
        score_cols = [c for c in score_cols if c in sub.columns]

        if donor_col and donor_col in sub.columns:
            ddf   = donor_mean(sub, donor_col, score_cols)
            level = 'donor'
        else:
            ddf   = sub[score_cols].copy()
            level = 'cell'

        for mes_sc in mes_score_cols:
            if mes_sc not in ddf.columns:
                continue
            r = spearman_ci(ddf['tolerance_positioning'], ddf[mes_sc])
            rows.append({
                'dataset':         ds,
                'mito_thresh_pct': thr,
                'level':           level,
                'n_cells_kept':    n_kept,
                'N_unit':          r['n'],
                'MES':             mes_sc.replace('_score',''),
                'r':               r['r'],
                'p':               r['p'],
                'ci_lo':           r['ci_lo'],
                'ci_hi':           r['ci_hi'],
            })
    gc.collect()

df = pd.DataFrame(rows)
assert not df.empty, 'No rows produced. Check that scored h5ads contain MES score columns.'
df['q_BH'] = bh_fdr(df['p'].fillna(1).values)


# 4) Pivot and summary statistics

pivot = df.pivot_table(
    index=['dataset','MES'], columns='mito_thresh_pct', values='r', aggfunc='mean'
).reset_index()
for thr in MITO_THRESHOLDS:
    if thr in pivot.columns:
        pivot.rename(columns={thr: f'rho_mt{thr}pct'}, inplace=True)

for thr in [25, 30, 35]:
    if 'rho_mt20pct' in pivot.columns and f'rho_mt{thr}pct' in pivot.columns:
        pivot[f'delta_vs_20pct_{thr}'] = pivot[f'rho_mt{thr}pct'] - pivot['rho_mt20pct']

delta_cols = [c for c in pivot.columns if c.startswith('delta_')]
if delta_cols:
    all_d = pivot[delta_cols].abs().values.ravel()
    all_d = all_d[np.isfinite(all_d)]
    log(f"  Max    |Delta_rho|: {np.max(all_d):.6f}")
    log(f"  Median |Delta_rho|: {np.median(all_d):.6f}")
    if 'delta_vs_20pct_35' in pivot.columns:
        med35 = pivot['delta_vs_20pct_35'].abs().median()
        log(f"  Median |Delta_rho| 35%% vs 20%% (manuscript <0.001): {med35:.6f}")

save_xlsx({"long": df, "pivot": pivot},
          TAB_DIR / "Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx")


# 5) Figure

datasets = df['dataset'].unique().tolist()
n_ds  = len(datasets)
n_col = min(n_ds, 4)
n_row = int(np.ceil(n_ds / n_col))

fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 3.5, n_row * 3.2), squeeze=False)

for idx, ds in enumerate(datasets):
    ax  = axes[idx // n_col][idx % n_col]
    sub = df[df['dataset'] == ds]
    mat = sub.pivot(index='MES', columns='mito_thresh_pct', values='r')
    mes_order = [m for m in MES_COLS if m in mat.index]
    mat = mat.reindex(index=mes_order)
    mat.columns = [f'MT<={c}%' for c in mat.columns]
    im = ax.imshow(mat.values.astype(float), cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, fontsize=7)
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=7)
    ax.set_title(ds.replace('_',' '), fontsize=8, pad=4)
    for ri in range(mat.shape[0]):
        for ci in range(mat.shape[1]):
            v = mat.values[ri, ci]
            if np.isfinite(v):
                ax.text(ci, ri, f'{v:.3f}', ha='center', va='center', fontsize=6,
                        color='white' if abs(v) > 0.4 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.7, label="Spearman rho", pad=0.02)

for idx in range(n_ds, n_row * n_col):
    axes[idx // n_col][idx % n_col].set_visible(False)

fig.suptitle('MES-Tolerance Correlation: MT% QC Threshold Sensitivity', fontsize=9, y=1.01)
plt.tight_layout()
save_fig(FIG_DIR / "Supplementary_FigS24_Mito_Threshold_Sensitivity.png", fig)


# Done

log("NB8 COMPLETED")
log(f"  Tables:  {TAB_DIR}")
log(f"  Figures: {FIG_DIR}")
log("  Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx")
log("    long  — rho, CI, p, q_BH, n_cells_kept per (dataset, MES, threshold)")
log("    pivot — rho at MT<=20/25/30/35% with delta vs 20% columns")
log("  Supplementary_FigS24_Mito_Threshold_Sensitivity.png")
